In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

In [13]:
# Prepare and tokenize dataset
#dataset = load_dataset("bigbio/pubmed_qa")
dataset = load_dataset("yelp_review_full")


Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset yelp_review_full downloaded and prepared to /Users/beto/.cache/huggingface/datasets/yelp_review_full/yelp_review_full/1.0.0/e8e18e19d7be9e75642fc66b198abadb116f73599ec89a69ba5dd8d1e57ba0bf. Subsequent calls will reuse this data.


  0%|          | 0/2 [00:00<?, ?it/s]

In [10]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
device = torch.device("mps")#("cpu")#("cuda")

tokenizer = GPT2Tokenizer.from_pretrained("stanford-crfm/BioMedLM")
#model = GPT2LMHeadModel.from_pretrained("stanford-crfm/BioMedLM").to(device)

In [11]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

In [17]:
tokenized_datasets = dataset.map(tokenize_function, batched=True)

small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(200))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(200))

Map:   0%|          | 0/650000 [00:00<?, ? examples/s]

Using pad_token, but it is not set yet.


ValueError: Asking to pad but the tokenizer does not have a padding token. Please select a token to use as `pad_token` `(tokenizer.pad_token = tokenizer.eos_token e.g.)` or add a new pad token via `tokenizer.add_special_tokens({'pad_token': '[PAD]'})`.

In [16]:
# Pass the appropriate abbreviation as the second argument
glue_metric = evaluate.load('glue', 'cola')
# Human generated desired targets
references = [1, 0]
# Predictions given by your model
predictions = [0, 1]
results = glue_metric.compute(predictions=predictions, references=references)
print(results)

# The result is -1 as the predictions are inverse of each other
Output: {'matthews_correlation': -1.0}

{'matthews_correlation': -1.0}
